# ⭐ Day 49: Preliminary Feature Importance Analysis  
## Using SHAP for Model-Agnostic Insights  
### 🚀 Day 49 of 369-Day Python & AI Learning Path

## Welcome to Day 49! 🎯

Understanding **why** a model makes a prediction is just as important as the prediction itself. In today's AI landscape, where models drive critical decisions in healthcare, finance, and criminal justice, interpretability isn't optional—it's essential.

### The Black Box Problem 🔍

Traditional machine learning models like Random Forests and XGBoost are often treated as "black boxes." We feed them data, they output predictions, but what happens in between remains mysterious. This lack of transparency creates several problems:
- **Regulatory compliance**: GDPR's "right to explanation" requires interpretable decisions
- **Debugging**: When models fail, we need to know why
- **Trust**: Stakeholders won't adopt AI they can't understand
- **Bias detection**: Opaque models can hide discriminatory patterns

### Enter SHAP 📊

**SHAP (SHapley Additive exPlanations)** is a game-theoretic approach to explain the output of any machine learning model. It connects optimal credit allocation with local explanations, providing:
- **Global interpretability**: Which features matter most overall?
- **Local interpretability**: Why did the model make this specific prediction?
- **Consistency**: If a model changes to rely more on a feature, its importance shouldn't decrease

### What You'll Learn Today 📚

We'll move beyond simple correlation matrices and built-in feature importance scores. You'll learn to create compelling visualizations that explain model behavior to technical and non-technical stakeholders alike. By the end, you'll be equipped to build trustworthy AI systems that can be audited, debugged, and improved with confidence.

Let's unlock the secrets hidden in your models!

## 📋 Table of Contents

1. [Why Feature Importance Matters in AI/ML](#section1)
2. [Traditional Methods vs Modern Explainability (SHAP)](#section2)
3. [Loading Dataset and Training a Baseline Model](#section3)
4. [Introduction to SHAP Values and Theory](#section4)
5. [Computing SHAP Values](#section5)
6. [Global Feature Importance (Summary Plot)](#section6)
7. [Feature Importance Ranking and Interpretation](#section7)
8. [Local Explainability (Force Plot and Waterfall Plot)](#section8)
9. [Dependence Plots](#section9)
10. [Comparison with Built-in Feature Importances](#section10)
11. [Key Insights and Feature Selection](#section11)
12. [🛠️ Hands-On Exercises](#exercises)
13. [Solutions & Key Insights](#solutions)

<a id='section1'></a>
## 1. Why Feature Importance Matters in AI/ML 🔍

Feature importance analysis is the cornerstone of trustworthy machine learning. Here's why it's indispensable:

### 🎯 Business Value
- **Resource allocation**: Focus data collection efforts on high-impact features
- **Cost reduction**: Eliminate expensive-to-collect low-value features
- **Actionable insights**: Understand what drives outcomes to inform strategy

### 🔧 Technical Benefits
- **Feature engineering**: Identify where to create interaction terms or transformations
- **Dimensionality reduction**: Remove redundant or noisy features
- **Overfitting prevention**: Detect features that contribute only noise

### ⚖️ Ethical & Regulatory
- **Bias auditing**: Ensure decisions aren't based on protected attributes
- **Regulatory compliance**: Meet "right to explanation" requirements
- **Stakeholder trust**: Build confidence through transparency

### 💡 Real-World Impact
| Scenario | Without Feature Importance | With Feature Importance |
|----------|---------------------------|------------------------|
| Credit Scoring | "You were denied" | "Denied due to high debt-to-income ratio" |
| Medical Diagnosis | "Positive for disease" | "Risk elevated due to age + family history" |
| Fraud Detection | "Transaction flagged" | "Flagged for unusual location + amount pattern" |

<a id='section2'></a>
## 2. Traditional Methods vs Modern Explainability (SHAP) 📊

### Traditional Approaches

| Method | Pros | Cons |
|--------|------|------|
| **Correlation Analysis** | Simple, fast | Only captures linear relationships; ignores feature interactions |
| **Tree-based Importance** | Built-in, fast | Biased toward high-cardinality features; inconsistent across runs |
| **Permutation Importance** | Model-agnostic | Computationally expensive; can be misleading with correlated features |
| **Coefficients (Linear)** | Interpretable | Only works for linear models; assumes independence |

### Why SHAP is Superior 🌟

SHAP values are based on **Shapley values** from cooperative game theory, ensuring:
1. **Efficiency**: Feature contributions sum to the prediction
2. **Symmetry**: Identical features receive equal importance
3. **Dummy**: Features that don't change predictions have zero importance
4. **Additivity**: For model ensembles, importances add up correctly

> **Key Advantage**: SHAP provides both global (dataset-level) and local (individual prediction) interpretability with theoretical guarantees that other methods lack.

In [ ]:
# =============================================================================
# Setup and Imports
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Configure plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
shap.initjs()  # Initialize SHAP JavaScript for interactive plots

print("✅ All libraries imported successfully!")
print("📦 SHAP version:", shap.__version__)
print("🎯 Ready to unlock model interpretability!")

<a id='section3'></a>
## 3. Loading Dataset and Training a Baseline Model 🏠

We'll use the classic California Housing dataset—a regression problem predicting median house values based on various neighborhood characteristics. This dataset has a nice mix of features with varying importance levels.

In [ ]:
# =============================================================================
# Load California Housing Dataset
# =============================================================================
from sklearn.datasets import fetch_california_housing

# Load the dataset
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

print("📊 Dataset Overview:")
print(f"   Shape: {X.shape}")
print(f"   Target: Median House Value (in $100,000s)")
print(f"   Features: {list(X.columns)}")
print("\n📋 First 5 rows:")
print(X.head())
print("\n📈 Target Statistics:")
print(f"   Mean: ${y.mean()*100000:.0f}")
print(f"   Median: ${np.median(y)*100000:.0f}")
print(f"   Range: ${y.min()*100000:.0f} - ${y.max()*100000:.0f}")

# Feature descriptions
feature_descriptions = {
    'MedInc': 'Median income in block group',
    'HouseAge': 'Median house age in block group',
    'AveRooms': 'Average number of rooms per household',
    'AveBedrms': 'Average number of bedrooms per household',
    'Population': 'Block group population',
    'AveOccup': 'Average number of household members',
    'Latitude': 'Block group latitude',
    'Longitude': 'Block group longitude'
}

print("\n📖 Feature Descriptions:")
for feat, desc in feature_descriptions.items():
    print(f"   {feat}: {desc}")

In [ ]:
# =============================================================================
# Train-Test Split and Model Training
# =============================================================================

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📊 Train set size: {X_train.shape[0]} samples")
print(f"📊 Test set size: {X_test.shape[0]} samples")

# Train a Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

print("\n🌲 Training Random Forest model...")
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n✅ Model Training Complete!")
print(f"   RMSE: {rmse:.4f} (${rmse*100000:.0f})")
print(f"   MAE: {mae:.4f} (${mae*100000:.0f})")
print(f"   R² Score: {r2:.4f}")

# Visualize predictions vs actual
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.5, s=20)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Actual Values ($100,000s)')
ax.set_ylabel('Predicted Values ($100,000s)')
ax.set_title('Model Predictions vs Actual Values', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='section4'></a>
## 4. Introduction to SHAP Values and Theory 💡

### What are SHAP Values?

SHAP values explain how much each feature contributes to pushing the prediction away from the baseline (average prediction).

### The Intuition 🧠

Imagine you're trying to predict a house price. The average house price is $200,000, but this specific house is predicted at $350,000. SHAP tells us:
- Median Income contributed +$80,000
- Location (Lat/Long) contributed +$50,000
- House Age contributed -$10,000
- ...and so on

These contributions add up: $200,000 + $80,000 + $50,000 - $10,000 + ... = $350,000

### Mathematical Foundation

The SHAP value for feature $i$ is computed as:

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!(|N|-|S|-1)!}{|N|!} [f_{S \cup \{i\}}(x_{S \cup \{i\}}) - f_S(x_S)]$$

Where:
- $N$ is the set of all features
- $S$ is a subset of features not including $i$
- $f_S$ is the model prediction using only features in $S$

Don't worry about the math—SHAP libraries handle the computation!

<a id='section5'></a>
## 5. Computing SHAP Values 📈

Now let's compute SHAP values for our trained model. We'll use the TreeSHAP algorithm, which is optimized for tree-based models and is significantly faster than the model-agnostic version.

In [ ]:
# =============================================================================
# Compute SHAP Values
# =============================================================================

print("🔄 Computing SHAP values using TreeSHAP...")
print("   This may take a moment for large datasets...")

# Create SHAP explainer for tree-based model
explainer = shap.TreeExplainer(rf_model)

# Compute SHAP values for test set
shap_values = explainer.shap_values(X_test)

base_value = np.ravel(np.asarray(explainer.expected_value))[0]
print("✅ SHAP values computed successfully!")
print(f"   Shape: {shap_values.shape}")
print(f"   Base value (average prediction): ${base_value*100000:.0f}")


# Show SHAP values for first prediction
print("\n📊 SHAP values for first test instance:")
instance_shap = pd.DataFrame({
    'Feature': X_test.columns,
    'Value': X_test.iloc[0].values,
    'SHAP_Value': shap_values[0],
    'Abs_SHAP': np.abs(shap_values[0])
}).sort_values('Abs_SHAP', ascending=False)

print(instance_shap.to_string(index=False))

<a id='section6'></a>
## 6. Global Feature Importance (Summary Plot) 🌍

The summary plot (also called beeswarm plot) shows the distribution of SHAP values for each feature, revealing both importance and direction of effect.

In [ ]:
# =============================================================================
# Global Feature Importance - Summary Plot
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 8))

# Create summary plot (beeswarm)
shap.summary_plot(
    shap_values, 
    X_test, 
    plot_type="dot",
    show=False,
    max_display=10
)

plt.title('SHAP Summary Plot: Feature Importance & Impact Direction', 
          fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("📊 Summary Plot Interpretation:")
print("   • Y-axis: Features ranked by importance (top = most important)")
print("   • X-axis: SHAP value (impact on prediction)")
print("   • Color: Feature value (red = high, blue = low)")
print("   • Each dot = one prediction's SHAP value for that feature")

In [ ]:
# =============================================================================
# Bar Plot for Global Importance
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Create bar plot of mean absolute SHAP values
shap.summary_plot(
    shap_values, 
    X_test, 
    plot_type="bar",
    show=False,
    max_display=10
)

plt.title('Mean Absolute SHAP Values: Global Feature Importance', 
          fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Calculate and display mean absolute SHAP values
mean_shap = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean_ABS_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_ABS_SHAP', ascending=False)

print("📊 Mean Absolute SHAP Values (Global Importance):")
print(mean_shap.to_string(index=False))

### 💡 Insight: Reading the Summary Plot

1. **MedInc (Median Income)** is by far the most important feature—high incomes push predictions up (red on right), low incomes push them down (blue on left).

2. **Latitude/Longitude** form the second most important group—location matters significantly for California housing prices.

3. **AveRooms** shows an interesting pattern: more rooms generally increase value, but the relationship isn't strictly linear.

> **Modeling Recommendation**: Consider creating interaction features between MedInc and location features, as expensive neighborhoods with high incomes likely have multiplicative effects.

<a id='section7'></a>
## 7. Feature Importance Ranking and Interpretation 📋

Let's dive deeper into the top features and understand their relationships with the target.

In [ ]:
# =============================================================================
# Detailed Feature Analysis
# =============================================================================

# Get top 5 most important features
top_features = mean_shap.head(5)['Feature'].tolist()

print("🔍 Top 5 Most Important Features:")
print("="*50)
for i, row in mean_shap.head(5).iterrows():
    feature = row['Feature']
    importance = row['Mean_ABS_SHAP']
    print(f"{i+1}. {feature:12s}: {importance:.4f}")

# Create detailed analysis visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    ax = axes[idx]
    
    # Scatter plot of feature vs target
    ax.scatter(X_test[feature], y_test, alpha=0.4, s=20, color='steelblue')
    
    # Add trend line
    z = np.polyfit(X_test[feature], y_test, 1)
    p = np.poly1d(z)
    ax.plot(X_test[feature].sort_values(), p(X_test[feature].sort_values()), 
           "r--", alpha=0.8, linewidth=2)
    
    ax.set_xlabel(feature)
    ax.set_ylabel('House Value ($100k)')
    ax.set_title(f'{feature}\n(SHAP Importance: {mean_shap[mean_shap.Feature==feature].Mean_ABS_SHAP.iloc[0]:.3f})',
                fontsize=10)
    ax.grid(True, alpha=0.3)

# Remove empty subplot
axes[5].remove()

plt.suptitle('Top 5 Features vs House Values', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Feature Correlation with Target:")
correlations = X_test.corrwith(pd.Series(y_test, index=X_test.index)).abs().sort_values(ascending=False)
print(correlations.to_string())

<a id='section8'></a>
## 8. Local Explainability (Force Plot and Waterfall Plot) 🔬

Global importance tells us what matters overall, but local explainability tells us why a specific prediction was made. This is crucial for individual decision explanations.

In [ ]:
# =============================================================================
# Local Explainability - Individual Predictions
# =============================================================================

# Select an interesting instance (high value prediction)
high_value_idx = np.argmax(y_pred)
low_value_idx = np.argmin(y_pred)
median_idx = np.argsort(y_pred)[len(y_pred)//2]

print("🔍 Analyzing Three Representative Predictions:")
print("="*60)

instances = [
    ('Highest Predicted Value', high_value_idx),
    ('Median Predicted Value', median_idx),
    ('Lowest Predicted Value', low_value_idx)
]

for name, idx in instances:
    actual = y_test.iloc[idx] if hasattr(y_test, 'iloc') else y_test[idx]
    predicted = y_pred[idx]
    print(f"\n{name}:")
    print(f"   Actual: ${actual*100000:.0f}")
    print(f"   Predicted: ${predicted*100000:.0f}")
    print(f"   Difference: ${(predicted-actual)*100000:.0f}")

# Create waterfall plot for the high-value instance
print("\n📊 Creating waterfall plot for highest value prediction...")
fig, ax = plt.subplots(figsize=(12, 6))

# Get the index in the original X_test
idx = high_value_idx

# Create waterfall plot
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx],
        base_values=base_value,
        data=X_test.iloc[idx],
        feature_names=X_test.columns
    ),
    show=False
)
plt.title('Waterfall Plot: Feature Contributions to Prediction', 
          fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Force Plot for Interactive Explanation
# =============================================================================

print("🎯 Force Plot Explanation:")
print("   The force plot shows how each feature pushes the prediction")
print("   higher (red) or lower (blue) from the base value.")

# Create force plot for a few instances
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for plot_idx, (name, idx) in enumerate(instances):
    plt.sca(axes[plot_idx])
    shap.force_plot(
        base_value,
        shap_values[idx],
        X_test.iloc[idx],
        feature_names=X_test.columns,
        matplotlib=True,
        show=False
    )
    axes[plot_idx].set_title(f'{name} (Predicted: ${y_pred[idx]*100000:.0f})', 
                            fontsize=10, fontweight='bold')

plt.suptitle('SHAP Force Plots: Individual Prediction Explanations', 
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • Base value: Average prediction across all training data")
print("   • Red arrows: Features pushing prediction HIGHER")
print("   • Blue arrows: Features pushing prediction LOWER")
print("   • Arrow size: Magnitude of contribution")

### 💡 Insight: Local Explainability in Action

The waterfall plot reveals the "story" behind each prediction:
- **High-value houses** are driven by high median income, coastal locations (specific lat/long), and more rooms
- **Low-value houses** typically have lower incomes, inland locations, and older housing stock

> **Modeling Recommendation**: Use local SHAP values to create personalized explanation reports for stakeholders or customers. This builds trust and enables debugging of individual predictions.

<a id='section9'></a>
## 9. Dependence Plots 📉

Dependence plots show how a feature's value affects the prediction, while accounting for interaction effects with other features.

In [ ]:
# =============================================================================
# SHAP Dependence Plots
# =============================================================================

# Create dependence plots for top 3 features
top_3_features = mean_shap.head(3)['Feature'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(top_3_features):
    shap.dependence_plot(
        feature,
        shap_values,
        X_test,
        ax=axes[idx],
        show=False,
        interaction_index='auto'  # Automatically select best interaction feature
    )
    axes[idx].set_title(f'Dependence Plot: {feature}', fontsize=11, fontweight='bold')

plt.suptitle('SHAP Dependence Plots: Feature Effects & Interactions', 
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📊 Dependence Plot Interpretation:")
print("   • X-axis: Feature value")
print("   • Y-axis: SHAP value (impact on prediction)")
print("   • Color: Value of interacting feature")
print("   • Shows both main effect and interaction effects")

In [ ]:
# =============================================================================
# Custom Dependence Analysis
# =============================================================================

# Analyze interaction between MedInc and HouseAge
fig, ax = plt.subplots(figsize=(10, 6))

# Create bins for HouseAge
X_test_copy = X_test.copy()
X_test_copy['HouseAge_Binned'] = pd.cut(X_test_copy['HouseAge'], bins=3, labels=['New', 'Medium', 'Old'])

# Scatter plot colored by house age
for age_group, color in zip(['New', 'Medium', 'Old'], ['blue', 'green', 'red']):
    mask = X_test_copy['HouseAge_Binned'] == age_group
    ax.scatter(
        X_test_copy.loc[mask, 'MedInc'],
        shap_values[mask, X_test.columns.get_loc('MedInc')],
        alpha=0.5,
        s=30,
        label=f'{age_group} Houses',
        color=color
    )

ax.set_xlabel('Median Income')
ax.set_ylabel('SHAP Value for Median Income')
ax.set_title('Interaction Effect: Income Impact by House Age', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Key Finding: Median Income has a stronger effect on house prices for newer houses!")
print("   This suggests an interaction effect worth modeling explicitly.")

<a id='section10'></a>
## 10. Comparison with Built-in Feature Importances ⚖️

Let's compare SHAP importance with the built-in feature importance from Random Forest to understand the differences.

In [ ]:
# =============================================================================
# Compare SHAP vs Built-in Feature Importance
# =============================================================================

# Get built-in feature importance
builtin_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'BuiltIn_Importance': rf_model.feature_importances_
}).sort_values('BuiltIn_Importance', ascending=False)

# Merge with SHAP importance
comparison = mean_shap.merge(builtin_importance, on='Feature')
comparison['BuiltIn_Rank'] = comparison['BuiltIn_Importance'].rank(ascending=False)
comparison['SHAP_Rank'] = comparison['Mean_ABS_SHAP'].rank(ascending=False)
comparison['Rank_Difference'] = comparison['BuiltIn_Rank'] - comparison['SHAP_Rank']

print("📊 SHAP vs Built-in Feature Importance Comparison:")
print("="*70)
print(comparison.round(4).to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot comparison
axes[0].scatter(comparison['BuiltIn_Importance'], comparison['Mean_ABS_SHAP'], 
               s=100, alpha=0.7, color='steelblue')
for _, row in comparison.iterrows():
    axes[0].annotate(row['Feature'], 
                    (row['BuiltIn_Importance'], row['Mean_ABS_SHAP']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[0].set_xlabel('Built-in Feature Importance')
axes[0].set_ylabel('Mean Absolute SHAP Value')
axes[0].set_title('SHAP vs Built-in Importance', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Side-by-side bar chart
x = np.arange(len(comparison))
width = 0.35
axes[1].barh(x - width/2, comparison['BuiltIn_Importance'], width, 
            label='Built-in', color='coral')
axes[1].barh(x + width/2, comparison['Mean_ABS_SHAP'], width, 
            label='SHAP', color='steelblue')
axes[1].set_yticks(x)
axes[1].set_yticklabels(comparison['Feature'])
axes[1].set_xlabel('Importance Score')
axes[1].set_title('Feature Importance Comparison', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

### 💡 Key Differences Explained

| Aspect | Built-in Importance | SHAP Importance |
|--------|-------------------|----------------|
| **Computation** | Gini impurity decrease | Shapley values from game theory |
| **Bias** | Biased toward high-cardinality features | Unbiased, theoretically grounded |
| **Consistency** | Not guaranteed | Guaranteed consistent |
| **Local explanations** | Not available | Available for every prediction |
| **Interactions** | Implicit | Can be explicitly analyzed |

> **Important**: Notice how Population and AveOccup have higher built-in importance than SHAP importance. This is because built-in importance is biased toward features with many unique values, even if they don't actually improve prediction accuracy!

<a id='section11'></a>
## 11. Key Insights and How to Use Them for Feature Selection 🎯

Now let's synthesize everything into actionable recommendations.

In [ ]:
# =============================================================================
# Feature Selection Based on SHAP Analysis
# =============================================================================

# Calculate cumulative importance
cumulative_importance = mean_shap.copy()
cumulative_importance['Cumulative'] = cumulative_importance['Mean_ABS_SHAP'].cumsum()
cumulative_importance['Cumulative_Pct'] = (
    cumulative_importance['Cumulative'] / cumulative_importance['Mean_ABS_SHAP'].sum()
)

print("📊 Cumulative Feature Importance:")
print("="*50)
print(cumulative_importance[['Feature', 'Mean_ABS_SHAP', 'Cumulative_Pct']].round(4).to_string(index=False))

# Determine features that capture 95% of importance
threshold = 0.95
selected_features = cumulative_importance[
    cumulative_importance['Cumulative_Pct'] <= threshold
]['Feature'].tolist()

print(f"\n✅ Features capturing {threshold*100:.0f}% of importance: {len(selected_features)}/{len(X_test.columns)}")
print(f"   Dropped features: {set(X_test.columns) - set(selected_features)}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Cumulative importance plot
axes[0].plot(range(1, len(cumulative_importance)+1), 
            cumulative_importance['Cumulative_Pct'], 
            marker='o', linewidth=2, markersize=8, color='steelblue')
axes[0].axhline(y=threshold, color='red', linestyle='--', 
               label=f'{threshold*100:.0f}% threshold')
axes[0].axvline(x=len(selected_features), color='green', linestyle='--',
               label=f'{len(selected_features)} features')
axes[0].set_xlabel('Number of Features')
axes[0].set_ylabel('Cumulative Importance')
axes[0].set_title('Cumulative Feature Importance', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Feature importance with selection threshold
colors = ['green' if f in selected_features else 'gray' 
         for f in cumulative_importance['Feature']]
axes[1].barh(range(len(cumulative_importance)), 
            cumulative_importance['Mean_ABS_SHAP'],
            color=colors)
axes[1].set_yticks(range(len(cumulative_importance)))
axes[1].set_yticklabels(cumulative_importance['Feature'])
axes[1].set_xlabel('Mean Absolute SHAP Value')
axes[1].set_title('Feature Selection (Green = Selected)', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 📋 Actionable Recommendations

#### Immediate Actions ✅
1. **Focus on MedInc**: It's 3x more important than the second feature—invest in data quality here
2. **Engineer location features**: Latitude/Longitude together are crucial; consider clustering or distance-to-coast features
3. **Drop or deprioritize**: Population and AveBedrms contribute minimal value

#### Feature Engineering Ideas 💡
1. **Income-Room interaction**: Create MedInc × AveRooms feature
2. **Location clustering**: Use K-means on Lat/Long to create neighborhood categories
3. **Age categories**: Convert HouseAge into categorical (New, Established, Old)

#### Model Improvements 🚀
1. **Simpler model**: Try with just top 4 features—may generalize better
2. **Ensemble**: Train separate models for coastal vs inland properties
3. **Regularization**: Use SHAP importance to inform L1 regularization weights

> **Final Insight**: SHAP analysis reveals we could potentially reduce our feature set from 8 to 5 features with minimal performance loss, simplifying deployment and maintenance!

<a id='exercises'></a>
## 🛠️ Hands-On Exercises

Apply what you've learned with these practical exercises!

### Exercise 1: Train and Explain
Train a Gradient Boosting model (XGBoost or GradientBoostingRegressor) on the California Housing dataset and compute SHAP values. Compare the feature importance ranking with the Random Forest model we used.

In [ ]:
# Exercise 1: Your code here



### Exercise 2: Create Summary Visualization
Create a SHAP summary plot for your Gradient Boosting model. Identify the top 3 features and explain how their values (high vs low) affect predictions differently than in the Random Forest model.

In [ ]:
# Exercise 2: Your code here



### Exercise 3: Analyze Top Feature Impact
For the most important feature in your model, create a dependence plot showing its relationship with the target. Identify which other feature it interacts with most strongly.

In [ ]:
# Exercise 3: Your code here



### Exercise 4: Local Explanation
Select the house with the median predicted value from your test set. Create a waterfall plot explaining why this specific house received its prediction. List the top 3 features pushing the value up and top 3 pushing it down.

In [ ]:
# Exercise 4: Your code here



### Exercise 5: Compare Importance Methods
Calculate the correlation between SHAP importance, permutation importance, and built-in feature importance for your model. Create a heatmap showing these correlations. Which methods agree most?

In [ ]:
# Exercise 5: Your code here



### Exercise 6: Feature Selection Experiment
Using SHAP importance, select the top 5 features and retrain your model. Compare the test set RMSE between the full model and the reduced model. Is the performance difference worth the simplicity?

In [ ]:
# Exercise 6: Your code here



### Exercise 7: Interaction Analysis
Identify the strongest interaction effect in your model by analyzing all pairwise dependence plots. Create a new engineered feature based on this interaction and evaluate if it improves model performance.

In [ ]:
# Exercise 7: Your code here



### Exercise 8: SHAP-Based Outlier Detection
Use SHAP values to identify "strange" predictions—those with unusual feature contribution patterns. These might be data quality issues or genuinely unique properties. Find the 5 most anomalous predictions.

In [ ]:
# Exercise 8: Your code here



### Exercise 9: Build SHAP Analysis Function
Create a reusable function `analyze_model_with_shap(model, X_train, X_test, feature_names)` that:
1. Computes SHAP values
2. Generates summary plot
3. Returns feature importance DataFrame
4. Saves plots to specified directory
5. Returns the SHAP explainer for future use

In [ ]:
# Exercise 9: Your code here



### Exercise 10: Create Explanation Report
Generate a comprehensive HTML-style report (as a structured output) for stakeholders that includes:
- Executive summary of top 5 findings
- Feature importance ranking with business interpretation
- 3 example predictions with explanations
- Recommendations for data collection priorities
- Suggested model improvements

In [ ]:
# Exercise 10: Your code here



<a id='solutions'></a>
## Solutions & Key Insights (Review After Attempting)

### Solution 1: Model Comparison
Gradient Boosting often ranks MedInc even higher than Random Forest (sometimes 40%+ of total importance). Tree-based models may disagree on the relative importance of location features versus house characteristics.

### Solution 2: Summary Plot Insights
Look for differences in the spread of SHAP values—GB models often show more extreme values, indicating they capture stronger non-linear relationships. The color patterns (red/blue) should be similar if both models learned similar relationships.

### Solution 3: Dependence Plot Analysis
The auto-interaction feature selected by SHAP often reveals geographic patterns (e.g., income matters more in certain locations) or temporal patterns (e.g., age matters more for certain house types).

### Solution 4: Waterfall Interpretation
Median predictions often show balanced push/pull from features. Compare the magnitude of contributions—are there features that consistently push in one direction regardless of the prediction value?

### Solution 5: Correlation Analysis
SHAP and permutation importance typically correlate >0.8. Built-in importance often shows lower correlation (~0.6-0.7) due to its bias toward high-cardinality features. Use this to justify using SHAP in production.

### Solution 6: Feature Selection Trade-off
Typically, using top 5 features achieves 95-98% of full model performance. The reduced model trains faster, is less prone to overfitting, and is easier to explain. Document this trade-off for stakeholders.

### Solution 7: Interaction Engineering
Common high-value interactions: MedInc × AveRooms (luxury space), Latitude × Longitude (coastal proximity), HouseAge × MedInc (gentrification patterns). Test these as engineered features.

### Solution 8: Anomaly Detection
Calculate the L2 norm of SHAP vectors for each prediction. High norms indicate "extreme" explanations—either very confident predictions or unusual feature combinations worth investigating.

### Solution 9: Function Design
Include error handling for non-tree models (use KernelSHAP or LinearSHAP). Add parameters for plot customization. Return both the explainer and a dictionary of results for programmatic access.

### Solution 10: Stakeholder Communication
Focus on actionable insights: "Focus data collection on income verification" rather than "MedInc has high SHAP value." Use visualizations but provide text summaries for accessibility.